# 05. Block Floating Point (BFP), Microscaling (MXFP), and NVFP4

- **Category:** 13_precision_and_numerics
- **Lineage & Citations:** Wilkinson (1963, *Rounding Errors in Algebraic Processes*); OCP Consortium (2023, *Microscaling Formats (MX) Specification*); NVIDIA Architecture Whitepaper (2024, *Blackwell Ultra FP4 Tensor Core Architecture*).
- **Purpose:** Formulate Block Floating Point (BFP) shared-exponent quantization, the Open Compute Project (OCP) MXFP4 microscaling standard, and NVIDIA Blackwell two-level NVFP4 scaling.
- **Topic Connections:** `🔗 Upstream Prerequisites:` [04. FP8 E4M3 E5M2 FNUZ and Minifloats FP6 FP4](04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.ipynb) $\to$ `🔗 Downstream Syntheses:` [06. Stochastic Rounding and Master Weight Free BF16 Training](06_stochastic_rounding_and_master_weight_free_bf16_training.ipynb)

> [!WARNING]
> **Decision Boundary: When NOT to use it**  
> Do NOT use large block sizes ($K > 64$) when quantizing activation tensors with extreme emergent outliers (spatial outliers contaminate the shared exponent across too many valid features); do NOT omit two-level scaling when calibrating NVFP4 on distributed tensor-parallel layers.

## Mathematical Foundations & Microscaling Formulations

1. **Block Floating Point (BFP) Shared Exponent**:
   $$E_{\text{shared}} = \lfloor \log_2(\max_{i} |x_i|) \rfloor, \quad m_i = \operatorname{clamp}\left( \operatorname{round}\left( \frac{x_i}{2^{E_{\text{shared}}}} \cdot (2^{M-1} - 1) \right), \dots \right)$$
2. **OCP MXFP4 Power-of-Two Scaling**:
   $$S_{\text{block}} = 2^{e_{\text{scale}}}, \quad e_{\text{scale}} = \left\lceil \log_2\left( \frac{\max_{i} |x_i|}{6.0} \right) \right\rceil, \quad \tilde{x}_i = \operatorname{Quantize}_{\text{FP4}}(x_i / S_{\text{block}})$$
3. **NVIDIA Blackwell NVFP4 Two-Level Scaling**:
   $$\hat{x}_i = \tilde{x}_i \times S_{\text{local}} \times S_{\text{global}}, \quad S_{\text{global}} = \frac{\max(|\mathbf{X}|)}{6.0}, \quad S_{\text{local}} \in \text{E8M0}$$
4. **Outlier Containment**: Microscaling confines activation outliers to a single 16/32-element block, preserving fidelity across remaining channels.

In [ ]:
import sys, time
from pathlib import Path
import torch
from safetensors.torch import load_file

r = Path.cwd()
while not (r / "Cargo.toml").exists() and r != r.parent: r = r.parent
if str(r) not in sys.path: sys.path.insert(0, str(r))
w_path = r / "weights/13_precision_and_numerics/05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.safetensors"
weights = load_file(str(w_path))
print(f"Loaded {len(weights)} tensors from {w_path.name}")

In [ ]:
def linear_anchor(x, weight, bias):
    return x @ weight + bias

def quantize_minifloat_core(x, e_bits, m_bits, bias, max_val):
    x_abs = torch.abs(x)
    is_zero = (x_abs == 0.0)
    x_safe = torch.where(is_zero, torch.ones_like(x_abs), x_abs)
    e = torch.floor(torch.log2(x_safe))
    e_min_normal = 1.0 - float(bias)
    step = torch.where(e < e_min_normal, 2.0 ** (e_min_normal - float(m_bits)), 2.0 ** (e - float(m_bits)))
    q = torch.round(x_safe / step) * step
    q = torch.clamp(q, min=0.0, max=float(max_val))
    return torch.where(is_zero, torch.zeros_like(x), torch.sign(x) * q)

def block_floating_point_quantize(x, block_size=16, mantissa_bits=8):
    orig_shape = x.shape
    x_reshaped = x.reshape(-1, block_size)
    amax = torch.max(torch.abs(x_reshaped), dim=-1, keepdim=True).values
    is_zero = (amax == 0.0)
    amax_safe = torch.where(is_zero, torch.ones_like(amax), amax)
    shared_exp = torch.floor(torch.log2(amax_safe))
    shared_scale = 2.0 ** shared_exp
    norm_x = x_reshaped / shared_scale
    max_int = float((1 << (mantissa_bits - 1)) - 1)
    q_mantissa = torch.clamp(torch.round(norm_x * max_int), -max_int, max_int)
    dequant = (q_mantissa / max_int) * shared_scale
    dequant = torch.where(is_zero, torch.zeros_like(dequant), dequant)
    return dequant.reshape(orig_shape), shared_scale.reshape(-1)

def ocp_mxfp4_quantize(x, block_size=32):
    orig_shape = x.shape
    x_reshaped = x.reshape(-1, block_size)
    amax = torch.max(torch.abs(x_reshaped), dim=-1, keepdim=True).values
    is_zero = (amax == 0.0)
    amax_safe = torch.where(is_zero, torch.ones_like(amax), amax)
    exp_scale = torch.ceil(torch.log2(amax_safe / 6.0))
    scale_block = 2.0 ** exp_scale
    scale_block = torch.where(is_zero, torch.ones_like(scale_block), scale_block)
    elem_scaled = x_reshaped / scale_block
    elem_q = quantize_minifloat_core(elem_scaled, e_bits=2, m_bits=1, bias=1, max_val=6.0)
    dequant = elem_q * scale_block
    dequant = torch.where(is_zero, torch.zeros_like(dequant), dequant)
    return dequant.reshape(orig_shape), scale_block.reshape(-1)

def nvfp4_two_level_quantize(x, block_size=16):
    global_max = float(torch.max(torch.abs(x)))
    global_scale = max(1e-8, global_max / 6.0)
    x_glob = x / global_scale
    dequant_local, local_scales = ocp_mxfp4_quantize(x_glob, block_size=block_size)
    dequant = dequant_local * global_scale
    return dequant, global_scale, local_scales

In [ ]:
# 1. Linear anchor parity
y_anchor = linear_anchor(weights["test_input"], weights["weight"], weights["bias"])
assert torch.max(torch.abs(y_anchor - weights["test_output"])) < 1e-6

# 2. Block Floating Point Parity
bfp_dequant, bfp_scales = block_floating_point_quantize(weights["bfp_input"], block_size=16, mantissa_bits=8)
assert torch.max(torch.abs(bfp_dequant - weights["bfp_dequant_ref"])) < 1e-5
assert torch.max(torch.abs(bfp_scales - weights["bfp_scales_ref"])) < 1e-5

# 3. OCP MXFP4 Parity
mx_dequant, mx_scales = ocp_mxfp4_quantize(weights["mx_input"], block_size=32)
assert torch.max(torch.abs(mx_dequant - weights["mx_dequant_ref"])) < 1e-5
assert torch.max(torch.abs(mx_scales - weights["mx_scales_ref"])) < 1e-5

# 4. NVFP4 Two-Level Parity
nv_dequant, nv_glob, nv_loc = nvfp4_two_level_quantize(weights["nv_input"], block_size=16)
assert torch.max(torch.abs(nv_dequant - weights["nv_dequant_ref"])) < 1e-5
assert abs(nv_glob - float(weights["nv_global_scale"])) < 1e-5
assert torch.max(torch.abs(nv_loc - weights["nv_local_scales_ref"])) < 1e-5
print("All BFP, OCP MXFP4, and NVFP4 parity checks passed!")

In [ ]:
# Microbenchmark: BFP vs MXFP4 Quantization Latency
t0 = time.perf_counter()
for _ in range(500):
    _ = block_floating_point_quantize(weights["bfp_input"], block_size=16, mantissa_bits=8)
bfp_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"BFP quantization latency: {bfp_us:.2f} us")

t0 = time.perf_counter()
for _ in range(500):
    _ = ocp_mxfp4_quantize(weights["mx_input"], block_size=32)
mx_us = (time.perf_counter() - t0) / 500 * 1e6
print(f"OCP MXFP4 quantization latency: {mx_us:.2f} us")

In [ ]:
# Pathological Misuse: Tensor-wide FP4 vs Microscaled MXFP4 under Activation Outliers
torch.manual_seed(999)
clean_data = torch.randn(4, 32)
# Introduce a single severe outlier in block 0
outlier_data = clean_data.clone()
outlier_data[0, 5] = 50.0

# 1. Tensor-wide FP4 quantization (single scale for whole tensor)
tensor_scale = float(torch.max(torch.abs(outlier_data))) / 6.0
q_tensorwide = quantize_minifloat_core(outlier_data / tensor_scale, 2, 1, 1, 6.0) * tensor_scale

# 2. Microscaled MXFP4 (block size 32)
q_microscaled, _ = ocp_mxfp4_quantize(outlier_data, block_size=32)

# Measure reconstruction RMSE on the OTHER blocks (uncontaminated rows 1, 2, 3)
rmse_tensorwide = float(torch.sqrt(torch.mean((clean_data[1:] - q_tensorwide[1:]) ** 2)))
rmse_microscaled = float(torch.sqrt(torch.mean((clean_data[1:] - q_microscaled[1:]) ** 2)))

print(f"Uncontaminated blocks RMSE (Tensor-wide): {rmse_tensorwide:.4f}")
print(f"Uncontaminated blocks RMSE (MXFP4):       {rmse_microscaled:.4f}")
assert rmse_microscaled < rmse_tensorwide * 0.2, "MXFP4 must preserve fidelity in non-outlier blocks!"
print(f"Microscaling fidelity protection factor: {rmse_tensorwide / rmse_microscaled:.1f}x lower error")

## Algorithmic Takeaways & Cross-Domain Isomorphisms

### Key Takeaways
1. **Outlier Containment**: Microscaling confines rare high-magnitude activation spikes strictly to the local block, insulating the remaining $99\%$ of neural activations.
2. **Minimal Scale Overhead**: MXFP4 with block size 32 achieves a total memory footprint of only $4.25$ bits per element ($3.76\times$ memory compression relative to FP16).
3. **Two-Level Scaling**: NVFP4 combines global channel-level calibration with micro-block local scales, providing Blackwell Tensor Cores with the mathematical robustness required for billion-parameter training.

---
### Synchronized Track Navigation
- **Architectural Reference (Markdown):** [05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.md](../../markdown/13_precision_and_numerics/05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.md)
- **PyTorch 2.x Module:** [05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.py](../../pytorch2/13_precision_and_numerics/05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.py)
- **Functional JAX Module:** [05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.py](../../jax/13_precision_and_numerics/05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.py)
- **Native Rust Candle Track:** [05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.rs](../../candle/13_precision_and_numerics/05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.rs)
- **Hardware Kernel (CPU / WGSL):** [05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.rs](../../kernels/13_precision_and_numerics/05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.rs)
- **Golden SafeTensors:** [05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.safetensors](../../weights/13_precision_and_numerics/05_block_floating_point_bfp_microscaling_mxfp_and_nvfp4.safetensors)

---
**Curriculum Navigation:**  
&larr; [Previous: 04. FP8 E4M3 E5M2 FNUZ and Minifloats FP6 FP4](04_fp8_e4m3_e5m2_fnuz_and_minifloats_fp6_fp4.ipynb) | [Master Index](../../README.md#13_precision_and_numerics) | [Next: 06. Stochastic Rounding and Master Weight Free BF16 Training](06_stochastic_rounding_and_master_weight_free_bf16_training.ipynb) &rarr;